# TTLD-Net — Phase 7 Fast Ablation (Colab Pro)

Train lại M0→M4 trên Colab khi server GPU lỗi.

**Upload lên `My Drive/SDO_train/` trước:**
- `ttld_colab_code.zip` (tạo bằng `ttld-net/scripts/pack_colab.ps1`)
- `bstld_train.zip` (+ `.001`… nếu split)
- `bstld_val.zip` (+ split nếu có)

Runtime → **GPU** (A100/L4 khuyến nghị). Profile `--fast`.

In [ ]:
# CELL 1 — GPU check
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import torch
assert torch.cuda.is_available(), 'Bật GPU Runtime trước'
print('CUDA:', torch.cuda.get_device_name(0))
print('VRAM GB:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))

In [ ]:
# CELL 2 — Mount Drive
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/SDO_train'
OUT_DIR = f'{DRIVE_DIR}/ttld_outputs'
os.makedirs(OUT_DIR, exist_ok=True)
print('Drive files:', sorted(os.listdir(DRIVE_DIR)))

In [ ]:
# CELL 3 — Extract CODE -> /content/SDO (tự sửa layout zip)
import os, sys, zipfile

REPO = '/content/SDO'
TTLD = f'{REPO}/ttld-net'
code_zip = f'{DRIVE_DIR}/ttld_colab_code.zip'
assert os.path.isfile(code_zip), f'Missing {code_zip}'

if not os.path.isfile(f'{TTLD}/utils/colab_setup.py'):
    with zipfile.ZipFile(code_zip) as z:
        print('Zip preview:', z.namelist()[:10])
        z.extractall('/content')

bootstrap = TTLD if os.path.isdir(TTLD) else '/content/ttld-net'
if os.path.isdir(bootstrap):
    sys.path.insert(0, bootstrap)

from utils.colab_setup import setup_sdo_repo
REPO, TTLD = setup_sdo_repo(content_root='/content', drive_dir=DRIVE_DIR)
%cd {TTLD}


In [ ]:
# CELL 4 — Extract DATASET + symlink vào monorepo paths
import os, subprocess, time, glob, shutil, yaml

DATA = '/content/bstld'
os.makedirs(DATA, exist_ok=True)

def extract_7z_split(drive_dir, base_name, dest_dir):
    part1 = os.path.join(drive_dir, base_name + '.001')
    single = os.path.join(drive_dir, base_name)
    if os.path.exists(part1):
        first = part1
        print(f'  split: {base_name}.001...')
    elif os.path.exists(single):
        first = single
        print(f'  single: {base_name}')
    else:
        print(f'  SKIP missing {base_name}')
        return False
    if not os.path.exists('/usr/bin/7z'):
        subprocess.run(['apt-get', 'install', '-y', '-q', 'p7zip-full'], check=True)
    t0 = time.time()
    r = subprocess.run(['7z', 'x', first, f'-o{dest_dir}', '-y'], capture_output=True, text=True)
    print(f'  done {time.time()-t0:.0f}s code={r.returncode}')
    if r.returncode != 0:
        print(r.stderr[-400:])
    return r.returncode == 0

def find_named_dir(root, name):
    for base, dirs, files in os.walk(root):
        if os.path.basename(base) == name:
            return base
    return None

def find_endswith(root, suffix):
    suffix = suffix.replace('\\', '/').rstrip('/')
    for base, dirs, files in os.walk(root):
        b = base.replace('\\', '/')
        if b.endswith(suffix) and glob.glob(base + '/**/*.png', recursive=True):
            return base
    return None

if len(glob.glob(f'{DATA}/**/dataset_train_rgb/**/*.png', recursive=True)) < 1000:
    print('Extracting train...')
    extract_7z_split(DRIVE_DIR, 'bstld_train.zip', DATA)
else:
    print('Train already extracted')

if len(glob.glob(f'{DATA}/**/dataset_val_sample/**/*.png', recursive=True)) < 100:
    print('Extracting val...')
    extract_7z_split(DRIVE_DIR, 'bstld_val.zip', DATA)
else:
    print('Val already extracted')

train_root = find_named_dir(DATA, 'dataset_train_rgb')
val_root = find_named_dir(DATA, 'dataset_val_sample')
train_rgb = find_endswith(DATA, 'rgb/train')
val_rgb = find_endswith(DATA, 'rgb/val')
train_yaml = next((p for p in glob.glob(f'{DATA}/**/train.yaml', recursive=True)), None)

print('train_root', train_root)
print('val_root', val_root)
print('train_rgb', train_rgb)
print('val_rgb', val_rgb)
print('train_yaml', train_yaml)
assert train_root and val_root and train_rgb and val_rgb, 'Dataset không đủ — kiểm tra zip Drive'

DS = f'{REPO}/apps/worker/datasets'
os.makedirs(DS, exist_ok=True)

def force_symlink(src, dst):
    if os.path.lexists(dst):
        if os.path.islink(dst) or os.path.isfile(dst):
            os.unlink(dst)
        else:
            shutil.rmtree(dst)
    os.symlink(os.path.abspath(src), dst)
    print('link', dst, '->', src)

force_symlink(train_root, f'{DS}/dataset_train_rgb')
force_symlink(val_root, f'{DS}/dataset_val_sample')

ty = f'{DS}/dataset_train_rgb/train.yaml'
if not os.path.isfile(ty):
    assert train_yaml, 'Thiếu train.yaml trong zip'
    shutil.copy2(train_yaml, os.path.join(train_root, 'train.yaml'))
assert os.path.isfile(ty), 'train.yaml vẫn thiếu'
print('train.yaml OK')

bstld = {
    'path': DS,
    'train': 'dataset_train_rgb/rgb/train',
    'val': 'dataset_val_sample/rgb/val',
    'nc': 4,
    'names': {0: 'red', 1: 'yellow', 2: 'green', 3: 'off'},
}
with open(f'{DS}/bstld.yaml', 'w') as f:
    yaml.safe_dump(bstld, f, sort_keys=False)

print('train pngs', len(glob.glob(f'{DS}/dataset_train_rgb/**/*.png', recursive=True)))
print('val pngs', len(glob.glob(f'{DS}/dataset_val_sample/**/*.png', recursive=True)))
print('READY')

In [ ]:
# CELL 5 — Install deps
%cd /content/SDO/ttld-net
!pip install -q "numpy<2" pyyaml pillow opencv-python-headless matplotlib tensorboard tqdm ultralytics polars
import torch, ultralytics
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
print('ultralytics', ultralytics.__version__)

In [ ]:
# CELL 6 — Smoke test M4 (10 steps)
%cd /content/SDO/ttld-net
!python train.py --config configs/m4_full_ttld.yaml --output logs/ablation/m4_smoke --device 0 --fast --max-steps 10

In [ ]:
# CELL 7 — Continue from M2 (--proplus). Keep M0/M1 checkpoints.
# Code already on /content/SDO from M1 run — no need re-extract unless you changed zip.
%cd /content/SDO/ttld-net
!bash scripts/continue_m2.sh --proplus

In [ ]:
# CELL 8 — Sync results → Drive (tự extract lại code nếu runtime mất /content/SDO)
import os, shutil, glob, zipfile, sys

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

DRIVE_DIR = '/content/drive/MyDrive/SDO_train'
OUT_DIR = f'{DRIVE_DIR}/ttld_outputs'
os.makedirs(OUT_DIR, exist_ok=True)
code_zip = f'{DRIVE_DIR}/ttld_colab_code.zip'

def find_ttld():
    for p in ('/content/SDO/ttld-net', '/content/ttld-net'):
        if os.path.isfile(os.path.join(p, 'train.py')):
            return p
    for base, _, files in os.walk('/content'):
        if base.endswith('ttld-net') and 'train.py' in files:
            return base
    return None

TTLD = find_ttld()
if TTLD is None:
    assert os.path.isfile(code_zip), f'Missing {code_zip} — upload zip lên Drive'
    print('Runtime mất code — extracting', code_zip)
    with zipfile.ZipFile(code_zip) as z:
        print('zip preview:', z.namelist()[:8])
        z.extractall('/content')
    # Normalize layout -> /content/SDO/ttld-net
    found = find_ttld()
    assert found, 'Zip không chứa ttld-net/train.py'
    dst = '/content/SDO/ttld-net'
    os.makedirs('/content/SDO', exist_ok=True)
    if os.path.abspath(found) != os.path.abspath(dst):
        if os.path.isdir(dst):
            shutil.rmtree(dst)
        shutil.move(found, dst)
    TTLD = dst

print('TTLD root:', TTLD)
print('OUT_DIR:', OUT_DIR)

print('\n--- already on Drive ---')
for p in sorted(glob.glob(f'{OUT_DIR}/**/*', recursive=True))[:40]:
    if os.path.isfile(p):
        print(' ', p)

print('\n--- local dirs ---')
for rel in ('checkpoints', 'results', 'results/ablation', 'logs/ablation'):
    path = os.path.join(TTLD, rel)
    n = len(os.listdir(path)) if os.path.isdir(path) else 0
    print(f'  {rel}: exists={os.path.isdir(path)} items={n}')

def sync_dir(name, src):
    if not os.path.isdir(src) or not os.listdir(src):
        print('skip (missing/empty):', name)
        return 0
    dst = os.path.join(OUT_DIR, name)
    os.makedirs(dst, exist_ok=True)
    n = 0
    for root, _, files in os.walk(src):
        rel = os.path.relpath(root, src)
        target = dst if rel == '.' else os.path.join(dst, rel)
        os.makedirs(target, exist_ok=True)
        for f in files:
            shutil.copy2(os.path.join(root, f), os.path.join(target, f))
            n += 1
    print(f'synced {n} files: {src} -> {dst}')
    return n

copied = 0
copied += sync_dir('checkpoints', os.path.join(TTLD, 'checkpoints'))
copied += sync_dir('results', os.path.join(TTLD, 'results'))
copied += sync_dir('logs_ablation', os.path.join(TTLD, 'logs', 'ablation'))

if copied == 0:
    print('\nNOTE: local trống (runtime reset sau train).')
    print('Nếu trước đó đã sync thành công, xem file trong OUT_DIR ở trên.')
    print('Nếu chưa từng sync: phải train lại — ckpt chỉ nằm trên /content.')

print('\n--- metrics on Drive ---')
mets = sorted(glob.glob(f'{OUT_DIR}/results/ablation/*_metrics.json'))
for p in mets:
    print(p)
    print(open(p, encoding='utf-8').read()[:500])
if not mets:
    print('(chưa có *_metrics.json trên Drive)')

## Resume nếu disconnect

Chạy lại CELL 1–5, rồi chỉ train phần còn thiếu (bỏ M0), ví dụ:

```python
%cd /content/SDO/ttld-net
!python scripts/run_ablation.py --configs m3_topology m4_full_ttld --device 0 --proplus
```

Sau đó chạy CELL 8 sync Drive.